# Notebook 2: Spread Construction

Builds the physically-grounded crack spread margin series using yield fractions
derived from the Aspen HYSYS atmospheric CDU simulation.

## The key differentiator

**Generic 3:2:1 crack spread (textbook):**
```
Crack_321 = (2/3 × RBOB × 42) + (1/3 × HO × 42) − WTI
```

**This project — HYSYS-weighted margin:**
```
Margin = (yield_naphtha × RBOB × 42) + (yield_diesel × HO × 42) − WTI − utility_cost_per_bbl
```

Where `yield_naphtha` and `yield_diesel` come directly from a converged Aspen HYSYS CDU simulation
of a WTI Light crude assay (ExxonMobil EMTEC, Reference WTIL220Y, 2020).

**Source:** `../hysys/yield_output.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# HYSYS-DERIVED INPUTS
# Source: Aspen HYSYS CDU simulation, WTI Light assay
# Feed: 9,336 kg/h, 25°C, 200 kPa
# Assay: ExxonMobil EMTEC WTIL220Y (October 2020)
# ============================================================

# Product stream mass flows (kg/h) from converged HYSYS simulation
crude_feed_kg_h = 9336.0

yield_naphtha        = 319.7  / crude_feed_kg_h   # 0.0342 — maps to RBOB (RB=F)
yield_kerosene       = 1096.0 / crude_feed_kg_h   # 0.1174 — jet fuel (informational)
yield_diesel         = 1142.0 / crude_feed_kg_h   # 0.1223 — maps to Heating Oil (HO=F)
yield_ago            = 1172.0 / crude_feed_kg_h   # 0.1255 — atmospheric gas oil
yield_residue        = 5353.0 / crude_feed_kg_h   # 0.5733 — atmospheric residue

# Operating cost from HYSYS Activated Economics
utility_cost_per_bbl = 0.112   # USD/bbl

print('HYSYS-derived yield fractions:')
print(f'  Naphtha  (→ RBOB):        {yield_naphtha:.4f} ({yield_naphtha*100:.1f}%)')
print(f'  Kerosene (informational): {yield_kerosene:.4f} ({yield_kerosene*100:.1f}%)')
print(f'  Diesel   (→ Heating Oil): {yield_diesel:.4f} ({yield_diesel*100:.1f}%)')
print(f'  AGO:                      {yield_ago:.4f} ({yield_ago*100:.1f}%)')
print(f'  Residue:                  {yield_residue:.4f} ({yield_residue*100:.1f}%)')
print(f'\nUtility cost per barrel: ${utility_cost_per_bbl:.3f}/bbl')

In [ ]:
# Load price data
df = pd.read_csv('../data/raw_prices.csv', index_col=0, parse_dates=True)
print(f'Loaded {len(df)} trading days')
df.head()

In [ ]:
# Unit conversion: RBOB and HO are quoted $/gallon → multiply by 42 to get $/bbl
df['RBOB_bbl']    = df['RBOB']    * 42
df['HeatOil_bbl'] = df['HeatOil'] * 42

# ============================================================
# HYSYS-WEIGHTED CRACK SPREAD MARGIN
# This is the physically-grounded formula
# ============================================================
df['margin_hysys'] = (
      yield_naphtha * df['RBOB_bbl']
    + yield_diesel  * df['HeatOil_bbl']
    - df['WTI']
    - utility_cost_per_bbl
)

# Generic 3:2:1 crack spread for comparison
df['crack_321'] = (
      (2/3) * df['RBOB_bbl']
    + (1/3) * df['HeatOil_bbl']
    - df['WTI']
)

print('Margin series statistics ($/bbl):')
print(df[['margin_hysys', 'crack_321']].describe().round(2))

In [ ]:
# Plot both spreads for comparison
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

axes[0].plot(df.index, df['margin_hysys'], color='#2c3e50', linewidth=1, label='HYSYS-weighted margin')
axes[0].axhline(df['margin_hysys'].mean(), color='#e74c3c', linestyle='--', linewidth=1, label=f'Mean: ${df["margin_hysys"].mean():.2f}/bbl')
axes[0].set_ylabel('Margin ($/bbl)')
axes[0].set_title('HYSYS-Weighted Refinery Margin (physically-grounded)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df.index, df['crack_321'], color='#7f8c8d', linewidth=1, label='Generic 3:2:1 crack spread')
axes[1].axhline(df['crack_321'].mean(), color='#e74c3c', linestyle='--', linewidth=1, label=f'Mean: ${df["crack_321"].mean():.2f}/bbl')
axes[1].set_ylabel('Crack Spread ($/bbl)')
axes[1].set_title('Generic 3:2:1 Crack Spread (textbook benchmark)')
axes[1].set_xlabel('Date')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../data/spread_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Note: HYSYS spread uses conservative operating point yields (3.4% naphtha, 12.2% diesel).')
print('The generic 3:2:1 spread uses 66.7% gasoline and 33.3% heating oil — much higher product weighting.')

In [ ]:
# Save constructed spread data
df.to_csv('../data/spread_data.csv')
print('Saved spread data to ../data/spread_data.csv')